###Hàm đánh giá (heuristic) để chuyển BFS (tìm kiếm theo chiều rộng) sang 1 thuật toán tìm kiếm có hướng như A*. Hàm đánh giá sẽ ước lượng số bước tối thiểu cần thiết để đạt được trạng thái mục tiêu, giúp giảm số trạng thái cần kiểm tra


In [1]:
import random
from collections import deque
import heapq

# --- MOVES và FACES ---
MOVES = {
    'U': [2, 0, 3, 1, 8, 9, 6, 7, 16, 17, 10, 11, 12, 13, 14, 15, 20, 21, 18, 19, 4, 5, 22, 23],
    'R': [11, 1, 2, 5, 22, 12, 3, 8, 4, 9, 10, 15, 6, 13, 14, 21, 16, 17, 18, 19, 20, 0, 7, 23],
    'F': [16, 10, 2, 3, 0, 9, 6, 7, 1, 19, 12, 5, 8, 4, 14, 15, 13, 17, 18, 11, 20, 21, 22, 23],
    'D': [0, 1, 2, 3, 19, 5, 4, 7, 8, 9, 23, 10, 13, 14, 15, 12, 16, 17, 6, 18, 20, 21, 11, 22],
    'L': [0, 20, 23, 3, 4, 5, 6, 7, 8, 2, 16, 11, 12, 9, 19, 15, 17, 18, 10, 1, 14, 21, 22, 13],
    'B': [0, 1, 7, 22, 4, 5, 23, 15, 8, 9, 10, 11, 12, 13, 17, 18, 16, 3, 2, 19, 21, 6, 14, 20],
}

FACES = {
    'U': [0,1,2,3],
    'R': [4,5,6,7],
    'F': [8,9,10,11],
    'D': [12,13,14,15],
    'L': [16,17,18,19],
    'B': [20,21,22,23],
}

# --- tiện ích ---
def apply_perm(state, perm):
    return [state[i] for i in perm]

def move(state, m):
    """Áp dụng 1 bước xoay"""
    return apply_perm(state, MOVES[m])

# Trạng thái solved (goal): 6 mặt đồng màu
GOAL = (
    ['W']*4 + ['R']*4 + ['G']*4 +
    ['Y']*4 + ['O']*4 + ['B']*4
)

ALL_MOVES = list(MOVES.keys())

# --- Hàm đánh giá (heuristic) ---
def heuristic(state):
    """Ước lượng số bước tối thiểu bằng cách đếm số ô sai màu trên mỗi mặt"""
    mismatch = 0
    for face, indices in FACES.items():
        face_colors = [state[i] for i in indices]
        target_color = face_colors[0]  # Giả sử màu đầu tiên là màu chính của mặt
        mismatch += sum(1 for color in face_colors if color != target_color)
    return mismatch // 4  # Chia 4 vì mỗi bước xoay ảnh hưởng ít nhất 4 ô

# --- in cube ASCII ---
def print_cube(state):
    """In trạng thái Rubik 2x2 dạng ASCII (mỗi mặt 2x2)."""
    def face_str(face_idx):
        return [
            "".join(state[i][0] for i in face_idx[0:2]),
            "".join(state[i][0] for i in face_idx[2:4])
        ]
    U = face_str(FACES['U'])
    R = face_str(FACES['R'])
    F = face_str(FACES['F'])
    D = face_str(FACES['D'])
    L = face_str(FACES['L'])
    B = face_str(FACES['B'])
    print("    " + U[0])
    print("    " + U[1])
    for row in range(2):
        print(L[row] + " " + F[row] + " " + R[row] + " " + B[row])
    print("    " + D[0])
    print("    " + D[1])
    print()

# --- sinh scramble ---
def random_scramble(length=3):
    return [random.choice(ALL_MOVES) for _ in range(length)]

def apply_sequence(state, seq):
    s = state[:]
    for m in seq:
        s = move(s,m)
    return s

# --- A* solver ---
def a_star_solve(start):
    count = 0
    if start == GOAL:
        return [], count
    open_set = [(heuristic(start), 0, start, [])]  # (f_score, g_score, state, path)
    closed_set = set()
    parent_move = {}

    while open_set:
        f_score, g_score, current, path = heapq.heappop(open_set)
        current_tuple = tuple(current)

        if current == GOAL:
            return path, count

        if current_tuple in closed_set:
            continue

        closed_set.add(current_tuple)

        for m in ALL_MOVES:
            next_state = move(current, m)
            next_tuple = tuple(next_state)
            count += 1

            if next_tuple in closed_set:
                continue

            g_score_new = g_score + 1
            h_score = heuristic(next_state)
            f_score_new = g_score_new + h_score

            new_path = path + [m]
            heapq.heappush(open_set, (f_score_new, g_score_new, next_state, new_path))
            parent_move[next_tuple] = m

    return None, count

# --- Demo ---
if __name__ == "__main__":
    scramble = random_scramble(3)
    scrambled = apply_sequence(GOAL, scramble)
    print("Scramble:", " ".join(scramble))
    print("Scrambled cube:")
    print_cube(scrambled)

    solution, count = a_star_solve(scrambled)
    print("Number of states explored: ", count)
    if solution:
        print("Solution ({} moves):".format(len(solution)), " ".join(solution))
        solved = apply_sequence(scrambled, solution)
        print("Solved cube:")
        print_cube(solved)
    else:
        print("No solution found (unexpected).")

Scramble: L U R
Scrambled cube:
    GW
    WW
YB GO BY RB
GW OY BO RY
    RG
    OR

Number of states explored:  620340
Solution (9 moves): R R R U U U L L L
Solved cube:
    WW
    WW
OO GG RR BB
OO GG RR BB
    YY
    YY

